# trailrunner in five minutes

What is the impact of **1000 kg of Portland cement, in Denmark, in 2030?**

Ordinary practice answers that by modelling static unit processes.
trailrunner instead builds on computational process model.

Every flow crossing a model's boundary — what it makes, what it needs, what it
emits — is a concept from the hierarchical
[sentier vocabulary](https://vocab.sentier.dev). That is what lets two models
connect to each other.

Nothing here touches the network. The vocabulary lookups come from the committed
`examples/pyst_cache.json` and `examples/pyst_labels.json`, the background
datasets from `examples/background_pack.parquet`, and the parameters from the
committed parquet files beside this notebook.

In [1]:
import sys
from pathlib import Path

# Run from anywhere: nbconvert starts the kernel in the notebook's directory,
# a human might start it from the repository root.
EXAMPLES = Path.cwd() if (Path.cwd() / "showcase_models.py").exists() else Path.cwd() / "examples"
sys.path.insert(0, str(EXAMPLES))

from trailrunner import Demand, Flow
from trailrunner.models.cement import CEMENT
from trailrunner.resolution import PystLabels

DEMAND = Demand(
    flow=Flow(iri=CEMENT, location="DK", time=2030), amount=1000.0, unit="kg"
)

# What a flow *is* is an IRI in https://vocab.sentier.dev -- not a free-text
# name. The vocabulary also knows what that concept is called, and those names
# are cached beside this notebook, so every print below reads in English with
# no network and no token.
VOCAB = PystLabels(EXAMPLES / "pyst_labels.json", client=None)


def name(iri: str, width: int | None = None) -> str:
    """The vocabulary's name for a concept, else the IRI's last segment."""
    label = VOCAB.label(iri) or iri.rsplit("/", 1)[-1]
    if width is not None and len(label) > width:
        label = label[: width - 1] + "…"  # a column, not a claim: tree() prints it in full
    return label


print(DEMAND.amount, DEMAND.unit, DEMAND.flow.iri)
print("that IRI is:", name(DEMAND.flow.iri))
print("where:", DEMAND.flow.location, " when:", DEMAND.flow.time)

1000.0 kg https://vocab.sentier.dev/products/bonsai/2025.1/BONSAI2025.1/fi_37440
that IRI is: Portland cement, aluminous cement, slag cement and similar hydraulic cements, except in the form of clinkers
where: DK  when: 2030


## 1. What a model does

A [`Model`](../docs/api/model.md) has one method. It takes a
[`Demand`](../docs/api/flow.md) and returns a [`Result`](../docs/api/result.md), answering
three questions at once. What did I make, what do I need, what did I emit.

In [2]:
from trailrunner import LocationHierarchy, ParameterSet
from trailrunner.models import cement
from trailrunner.models.cement import CementPlant

HIERARCHY = LocationHierarchy({"CH": "RER", "DK": "RER", "FR": "RER", "RER": "GLO"})
cement_params = ParameterSet.from_parquet(
    EXAMPLES / "cement_params.parquet", hierarchy=HIERARCHY
)
cement_model = CementPlant(params=cement_params)

answer = cement_model.apply(DEMAND)

# where and when get their own columns: a flow's location and year are part of
# its identity, not an annotation on its name.
print(f"{'':>13}  {'amount':>8} {'unit':<4} {'flow':<44} {'where':<5} {'when':>5}")
for field in ("production", "technosphere", "biosphere"):
    for exchange in getattr(answer, field):
        flow = exchange.flow
        print(
            f"{field:>13}  {exchange.amount:>8.1f} {exchange.unit:<4} "
            f"{name(flow.iri, 44):<44} {flow.location:<5} {flow.time:>5}"
        )
print(f"{'provenance':>13}  {answer.provenance}")

                 amount unit flow                                         where  when
   production    1000.0 kg   Portland cement, aluminous cement, slag cem… DK     2030
 technosphere    1125.0 kg   Gypsum; anhydrite; limestone flux; limeston… DK     2030
 technosphere    2475.0 MJ   Natural gas, liquefied or in the gaseous st… DK     2030
 technosphere      10.0 kg   Quicklime, slaked lime and hydraulic lime    DK     2030
 technosphere     100.0 kWh  electricity                                  DK     2030
    biosphere     397.5 kg   co2-fossil                                   DK     2030
    biosphere     138.6 kg   co2-fossil                                   DK     2030
   provenance  {'location_requested': 'DK', 'location_used': 'DK', 'location_fallback': False, 'time_requested': 2030, 'time_used': 2030, 'time_interpolated': False, 'source': 'modelled'}


Every name printed there — `Quicklime, slaked lime and hydraulic lime`,
`co2-fossil` — is the vocabulary's label for a real concept.

Because the cement plant is a computation model, we can do additional calculations, taking into account the context the process is run in.
This data is retrieved from a set of parquet files based on their embedded metadata, created with [`trailpack`](https://github.com/TimoDiepers/trailpack).
For example, fuel demands could increase with higher ambient moisture and lower temperatures:

```python
penalty = moisture_penalty(row["moisture"], row["temperature"])
fuel = row["fuel_demand"] * clinker * penalty
```

The same 1000 kg, asked for in four different places and years:

In [3]:
print(f"{'where':>6} {'when':>6} {'moisture':>9} {'degC':>6} {'penalty':>9} {'gas [MJ]':>10}")
for location in ("DK", "RER"):
    for year in (2030, 2040):
        feed = cement_params.at(location=location, time=year)
        answer = cement_model.apply(
            Demand(flow=Flow(iri=CEMENT, location=location, time=year),
                   amount=1000.0, unit="kg")
        )
        gas = [d for d in answer.technosphere if d.flow.iri == cement.NATURAL_GAS][0]
        print(
            f"{location:>6} {year:>6} {feed['moisture']:>9.3f} {feed['temperature']:>6.1f} "
            f"{cement.moisture_penalty(feed['moisture'], feed['temperature']):>9.3f} "
            f"{gas.amount:>10.1f}"
        )

 where   when  moisture   degC   penalty   gas [MJ]
    DK   2030     0.040   10.0     1.000     2475.0
    DK   2040     0.040   11.0     0.996     2099.6
   RER   2030     0.060    9.0     1.044     2923.2
   RER   2040     0.055   10.0     1.030     2447.3


Not every process needs computing, though: where a process has real history —
years of stack-monitor readings, say — trailrunner reads that instead of
calculating it, and reaches for a computational model only where there is
none, such as a future year or a process that does not exist yet.

The plant has a stack monitor and years of historic readings.
`MeteredCementPlant` declares the same product IRI as `CementPlant` and a
[`Coverage`](../docs/api/coverage.md) that ends where the other one begins.
Nothing else changes: [`Glossary`](../docs/api/glossary.md)`.resolve` already
filters candidates by coverage, so the year on the demand decides which one
answers.

```python
coverage = Coverage(time_range=(2018, 2025))  # MeteredCementPlant
coverage = Coverage(time_range=(2026, 2050))  # CementPlant
```

In [4]:
from showcase_models import MODELS
from trailrunner import Glossary, Orchestrator

for year in (2023, 2030):
    run = Orchestrator(Glossary(MODELS)).calculate(
        Demand(flow=Flow(iri=CEMENT, location="DK", time=year), amount=1000.0, unit="kg")
    )
    root = run.nodes[0]
    direct = [e for e in root.result.biosphere if e.flow.iri == cement.CO2_FOSSIL]
    print(f"{year}  answered by {root.model}")
    print(f"      source: {run.provenance[root.id]['source']}")
    print(f"      direct CO2: {sum(e.amount for e in direct):>6.1f} kg "
          f"in {len(direct)} exchange(s)")
    for exchange in direct:
        print(f"          {exchange.amount:>6.1f} kg")

2023  answered by MeteredCementPlant
      source: measured
      direct CO2:  562.0 kg in 1 exchange(s)
           562.0 kg
2030  answered by CementPlant
      source: modelled
      direct CO2:  536.1 kg in 2 exchange(s)
           397.5 kg
           138.6 kg


Look at the biosphere flows above: 2023 has one, 2030 has two. The model
knows which kilogram came from the limestone and which from the flame,
because it computed them separately. The meter does not: a stack monitor sees
one plume and cannot tell you what made it.

Note what the metered model still sends upstream. Its gas, its lime and its
electricity are *inputs* — their emissions happen somewhere else — so they go
back on the queue and are answered by whoever supplies them, exactly as the
computed model's are. A meter at the fence line says nothing about what happens
beyond it.

The normative choices a study still has to make, and a model that co-produces,
are in [When a model makes two things](coproduction.ipynb) and
[Attribution](../docs/content/attribution.md).

## 2. Models find each other through a vocabulary

Every flow is identified by an IRI from the hierarchical
[sentier vocabulary](https://vocab.sentier.dev). A `Demand` for
`.../BONSAI2025.1/fi_37420` finds whoever declared that same IRI in `produces`,
with no name matching and no unit guessing in between. That is what lets two
models written by two people compose at all, and it is what the orchestrator uses
to walk outward.

```mermaid
%%{init: {'layout': 'elk'}}%%
flowchart TB
    D([initial demand]) --> Q
    Q[[Queue]]

    Q -->|pop demand| C{{ResolutionChain}}
    C -->|who offers?| G[(Glossary: available models)]
    G -->|Offer: model + demand| C

    C -->|nobody offers| L[(Log)]
    C -->|selected offer| R[Runner]

    R -->|apply demand| M[Model: your code]
    M -->|Result| R

    R -->|Result: technosphere demands| Q
    R -->|Result: biosphere flows| I[(inventory)]

    R --> L
    L --> P([Report])

    classDef resolution fill:#2dd4bf22,stroke:#2dd4bf
    classDef execution fill:#f59e0b22,stroke:#f59e0b
    classDef record fill:#8b5cf622,stroke:#8b5cf6
    class C,G resolution
    class R,M execution
    class L,P record
```

*Legend: resolution (teal), execution (amber), record (violet).*

`Orchestrator.calculate` is a `while queue:` and little else. Pop a demand, ask
the chain who can answer it, hand the offer to the [`Runner`](../docs/api/runner.md),
push the `Result`'s technosphere demands back on, write everything to the
[`Log`](../docs/api/log.md). Here's the queue for the cement example:

In [5]:
from showcase_models import MODELS  # the same list `trailrunner run --models` loads
from trailrunner import Glossary, Orchestrator
from trailrunner.resolution import ModelProvider, ResolutionChain


class Narrating(ResolutionChain):
    """A chain that says what it was asked. The Orchestrator takes any chain."""

    def offer(self, demand, exclude=()):
        offer = super().offer(demand, exclude=exclude)
        who = type(offer.model).__name__ if offer else "cutoff (nobody offered)"
        flow = demand.flow
        print(
            f"pop {demand.amount:>9.4g} {demand.unit:<4} {name(flow.iri, 32):<32} "
            f"{flow.location:<5} {flow.time:>5}  -> {who}"
        )
        return offer


tier1 = ModelProvider(Glossary(MODELS))
print(f"{'':>3} {'amount':>9} {'unit':<4} {'flow':<32} {'where':<5} {'when':>5}  -> answered by")
first = Orchestrator(Narrating([tier1])).calculate(DEMAND)

       amount unit flow                             where  when  -> answered by
pop      1000 kg   Portland cement, aluminous ceme… DK     2030  -> CementPlant
pop      1125 kg   Gypsum; anhydrite; limestone fl… DK     2030  -> cutoff (nobody offered)
pop      2475 MJ   Natural gas, liquefied or in th… DK     2030  -> NaturalGasSupply
pop        10 kg   Quicklime, slaked lime and hydr… DK     2030  -> cutoff (nobody offered)
pop       100 kWh  electricity                      DK     2030  -> GridElectricity
pop     68.75 Nm3  natural-gas-at-production        NO     2030  -> NaturalGasExtraction
pop     50.53 tkm  natural-gas-transport-offshore-… NO     2030  -> NaturalGasOffshorePipelineTransport
pop     8.466 kWh  electricity-natural-gas          DK     2030  -> GasPower
pop     84.66 kWh  electricity-wind                 DK     2030  -> cutoff (nobody offered)
pop      12.7 kWh  electricity-hydro                DK     2030  -> cutoff (nobody offered)
pop 8.995e-08 unit pipeline-natur

In [6]:
print(first.summary())
print()
print(first.tree(labels=VOCAB.label))  # the vocabulary's names, where it has one

11 nodes, 11 inventory entries
12 unresolved (no_model_found: 12)
0 proxies
attribution: allocation=none, capital=per_output

1000 kg Portland cement, aluminous cement, slag cement and similar hydraulic cements, except in the form of clinkers @DK/2030  [model: CementPlant]
  2475 MJ Natural gas, liquefied or in the gaseous state @DK/2030  [model: NaturalGasSupply]
    68.75 Nm3 natural-gas-at-production @NO/2030  [model: NaturalGasExtraction]
    50.5312 tkm natural-gas-transport-offshore-pipeline-long-distance @NO/2030  [model: NaturalGasOffshorePipelineTransport]
      0.0130625 Nm3 natural-gas-at-production @NO/2030  [model: NaturalGasExtraction]
      8.99456e-08 unit pipeline-natural-gas-long-distance-high-capacity-offshore @NO/2030  [cutoff: no_model_found]
      16.5404 MJ natural-gas-burned-in-gas-turbine @NO/2030  [cutoff: no_model_found]
      5.86162e-06 tkm transport-freight-lorry-16t-32t @NO/2030  [cutoff: no_model_found]
      5.86162e-05 kg disposal-used-mineral-oil-10-p

The fuel is no longer a leaf. `NaturalGasSupply` answers the kiln's 2475 MJ,
turns them into wellhead volume and route length, and hands those to a gas
field and to `NaturalGasOffshorePipelineTransport` — a model reverse-engineered
from the BAFU/ecoinvent pipeline datasets, which was registered in this list
long before anything asked it for a tonne-kilometre.

Note where the pipeline runs. The supply model places both demands at the
*origin*, so the Danish kiln's gas is transported in `NO` and its leakage is
priced at the Norwegian shelf's low-leakage tier, not at a Danish average that
does not exist. The pipeline's own inputs — compressor fuel, the pipe itself,
a maintenance lorry — are cutoffs, and they are in the report with a reason.


## 3. A demand nobody answers is relaxed along the vocabulary

[`ResolutionChain`](../docs/api/resolution.md) is a list of providers, asked in
order, and the first offer wins. Tier 1 is the computational models. Every later tier is a concession, and the tier that
made it writes what it conceded into the node's resolution.

**Tier 2 generalises the demand.** The plant blends in a little hydrated lime,
so it asks for `fi_37420`, "Quicklime, slaked lime and hydraulic lime". Nobody
produces it. One `skos:broader` step reaches `fi_3742` — spelled identically,
and produced by nobody either. The *second* step reaches `fi_374`, "Plaster,
lime and cement", and a supplier registered there can answer.

Notice what that concession costs. `fi_374` is an average over plaster, lime
**and cement** — so a lime demand was answered by a category containing the very
product this plant is making. It is the best answer available and a poor answer
in substance, and it is written at the node rather than lost.

**Tier 3 borrows a dataset.** Given its `Fleet`, `CementPlant` demands each
kiln's construction in the year that kiln was built, and a construction model
turns that into steel and aluminium taken from the curated background pack.

`BinderSupply` and `CementKilnConstruction` below are written in this notebook
rather than shipped, because nothing in the repository produces `fi_374`. Their
burdens and material intensities are invented. The `skos:broader` walk, the
step budget, the pack lookup, the completeness flag and the construction pulse
are the library, and every block of output below is what it actually printed.

In [7]:
from trailrunner import Exchange, Fleet, Model, Result
from trailrunner.core.settings import ALLOCATION_RULES
from trailrunner.models.cement import CEMENT_KILN, CO2_FOSSIL

BINDERS = "https://vocab.sentier.dev/products/bonsai/2025.1/BONSAI2025.1/fi_374"
STEEL = "https://vocab.sentier.dev/products/steel-low-alloyed"
ALUMINIUM = "https://vocab.sentier.dev/products/aluminium-primary"


class BinderSupply(Model):
    """Plaster, lime and cement, averaged. Illustrative burden.

    Registered two skos:broader steps above the lime the plant asks for, which
    is the point: it cannot answer a demand for lime, only the generalised
    demand that tier 2 makes out of it after the first step finds nobody.

    One consequence worth naming rather than leaving as a trap: a cement demand
    in a year *neither* CementPlant nor MeteredCementPlant covers would relax
    fi_37440 -> fi_3744 -> fi_374 and land here, because cement is also a
    binder. The tour never asks for such a year. A study that might should
    narrow this model's coverage rather than rely on that.
    """

    produces = [BINDERS]
    supports = ALLOCATION_RULES  # monofunctional

    co2_per_kg = 0.9  # kg CO2 per kg of binder, calcination and kiln fuel together

    def apply(self, demand):
        here = {"location": demand.flow.location, "time": demand.flow.time}
        return Result(
            production=[Exchange(flow=demand.flow, amount=demand.amount, unit=demand.unit)],
            technosphere=[],
            biosphere=[Exchange(flow=Flow(iri=CO2_FOSSIL, **here),
                                amount=demand.amount * self.co2_per_kg, unit="kg")],
            provenance={"binder_average": True},
        )


class CementKilnConstruction(Model):
    """What a kiln line is made of. Illustrative material intensities."""

    produces = [CEMENT_KILN]
    supports = ALLOCATION_RULES

    steel_per_capacity = 0.012      # kg steel per kg/year of clinker capacity
    aluminium_per_capacity = 0.0008  # kg aluminium, likewise

    def apply(self, demand):
        here = {"location": demand.flow.location, "time": demand.flow.time}
        return Result(
            production=[Exchange(flow=demand.flow, amount=demand.amount, unit=demand.unit)],
            technosphere=[
                Demand(flow=Flow(iri=STEEL, **here),
                       amount=demand.amount * self.steel_per_capacity, unit="kg"),
                Demand(flow=Flow(iri=ALUMINIUM, **here),
                       amount=demand.amount * self.aluminium_per_capacity, unit="kg"),
            ],
            biosphere=[],
        )

In [8]:
from trailrunner.resolution import (
    BackgroundPack, BackgroundProvider, GeneralisingProvider, PystTaxonomy,
)

# The kilns that were actually built. The 1985 line is past its lifetime by
# 2030, so the fleet leaves it out and the report never mentions it.
FLEET_ROWS = [
    {"kiln": "dk-old", "location": "DK", "build_year": 1985, "capacity": 250_000_000.0, "lifetime": 40.0},
    {"kiln": "dk-1", "location": "DK", "build_year": 2026, "capacity": 400_000_000.0, "lifetime": 40.0},
    {"kiln": "dk-2", "location": "DK", "build_year": 2029, "capacity": 800_000_000.0, "lifetime": 40.0},
]
fleet = Fleet(FLEET_ROWS, units={"capacity": "kg/year", "lifetime": "year"}, hierarchy=HIERARCHY)

MODELS_PLUS = [
    CementPlant(params=cement_params, fleet=fleet),
    *(model for model in MODELS if not isinstance(model, CementPlant)),
    BinderSupply(),
    CementKilnConstruction(),
]

tier1 = ModelProvider(Glossary(MODELS_PLUS))
# client=None: no network, ever. Every skos:broader answer comes from the file.
taxonomy = PystTaxonomy(EXAMPLES / "pyst_cache.json", client=None)
tier2 = GeneralisingProvider(tier1, hierarchy=HIERARCHY, taxonomy=taxonomy)
pack = BackgroundPack.from_parquet(EXAMPLES / "background_pack.parquet", hierarchy=HIERARCHY)
CHAIN = ResolutionChain([tier1, tier2, BackgroundProvider(pack)])

report = Orchestrator(CHAIN).calculate(DEMAND)
print(report.summary())
print()
print(report.tree(labels=VOCAB.label))

18 nodes, 13 inventory entries
11 unresolved (generalisation_exhausted: 11)
5 proxies (4 incomplete)
attribution: allocation=none, capital=per_output

1000 kg Portland cement, aluminous cement, slag cement and similar hydraulic cements, except in the form of clinkers @DK/2030  [model: CementPlant]
  2475 MJ Natural gas, liquefied or in the gaseous state @DK/2030  [model: NaturalGasSupply]
    68.75 Nm3 natural-gas-at-production @NO/2030  [model: NaturalGasExtraction]
    50.5312 tkm natural-gas-transport-offshore-pipeline-long-distance @NO/2030  [model: NaturalGasOffshorePipelineTransport]
      0.0130625 Nm3 natural-gas-at-production @NO/2030  [model: NaturalGasExtraction]
      8.99456e-08 unit pipeline-natural-gas-long-distance-high-capacity-offshore @NO/2030  [cutoff: generalisation_exhausted]
      16.5404 MJ natural-gas-burned-in-gas-turbine @NO/2030  [cutoff: generalisation_exhausted]
      5.86162e-06 tkm transport-freight-lorry-16t-32t @NO/2030  [cutoff: generalisation_exhaust

In [9]:
lime_node = [node for node in report.nodes if node.demand.flow.iri == cement.LIME][0]
for key, value in report.proxies[lime_node.id].items():
    print(f"{key:>12}: {value}")

print()
print("     asked, in words:", name(cement.LIME))
print("  answered, in words:", name(BINDERS))
print()
# The rung in between, which the walk passed through and nobody produces.
STEP_ONE = "https://vocab.sentier.dev/products/bonsai/2025.1/BONSAI2025.1/fi_3742"
print("one step up would be:", name(STEP_ONE), "-- same words, still nobody")

       model: BinderSupply
 relaxations: ['product: fi_37420 -> fi_374']
       asked: https://vocab.sentier.dev/products/bonsai/2025.1/BONSAI2025.1/fi_37420 @DK/2030
    answered: https://vocab.sentier.dev/products/bonsai/2025.1/BONSAI2025.1/fi_374 @DK/2030
        tier: generalising

     asked, in words: Quicklime, slaked lime and hydraulic lime
  answered, in words: Plaster, lime and cement

one step up would be: Quicklime, slaked lime and hydraulic lime -- same words, still nobody


Every concession is deliberate, ordered by the practitioner, and written down.

In [10]:
from trailrunner import viz
from trailrunner.assessment import Method, assess

# IPCC AR6 GWP100, stated here rather than read from a background database:
# the mapping from a gas to its warming potential is a fact about the gas.
GWP100 = Method(
    rows=[
        {"flow_iri": "https://vocab.sentier.dev/flows/co2-fossil", "flow_unit": "kg",
         "location": "GLO", "cf": 1.0},
        {"flow_iri": "https://vocab.sentier.dev/flows/ch4-fossil", "flow_unit": "kg",
         "location": "GLO", "cf": 29.8},
        {"flow_iri": "https://vocab.sentier.dev/flows/n2o", "flow_unit": "kg",
         "location": "GLO", "cf": 273.0},
    ],
    unit="kg CO2-eq",
    name="IPCC AR6 GWP100",
    hierarchy=HIERARCHY,
)

assessment = assess(report, GWP100)
sankey_figure = viz.sankey(report, assessment=assessment)
sankey_figure

## 4. Time rides along

Nothing in the loop was ever told about time. A [`Flow`](../docs/api/flow.md)
carries its year the way it carries its location, so every demand pushed, every emission accumulated and
every node logged is already dated. The kilns doing the calcining were built in
2026 and 2029. The cement, and the gas firing the kiln, happen in 2030.

So the inventory is a time series, and can be characterized as one.

In [11]:
from trailrunner.assessment import assess_dynamic

# No characterization table is passed: default_functions() covers co2-fossil
# and ch4-fossil, which is where the calcination CO2, the combustion CO2 and
# the pipeline's leaked methane are written. The uncharacterized count below
# is the rest of what the gas chain emits -- ethane, mercury, NMVOC, and the
# gas taken out of the ground -- which no climate method scores, and which is
# reported rather than dropped.
dynamic = assess_dynamic(report, metric="radiative_forcing", horizon=100)
print(dynamic.summary())

by_year = dynamic.series.groupby(dynamic.series["date"].dt.year)["amount"].sum()
print()
print("marginal radiative forcing, first years [W/m2]:")
print(by_year.head(6).to_string())

4.897e-11 W·yr/m2
metric: radiative_forcing, horizon: 100 years
horizon anchored at: 2026-01-01
18 uncharacterized exchanges
0 wrong unit exchanges
0 undated exchanges
0 beyond-horizon exchanges
11 unresolved
5 proxies

marginal radiative forcing, first years [W/m2]:
date
2027    2.973750e-17
2028    5.434073e-17
2029    2.520040e-17
2030    5.947499e-17
2031    9.120960e-13
2032    1.666609e-12


In [12]:
curve_figure = viz.curve(dynamic)
curve_figure

The faint bars are the per-year forcing and the red line is its running total.
The kilns show up in 2027 and 2029, and then 2031 arrives and the scale of the
plot changes: building two cement plants is four orders of magnitude
below one year of making cement in them. That is not a flaw in the example. It
is what the industry's problem actually looks like, and it is visible here only
because the dates survived.

A static score gives one number for all of that, and no way to ask when any of
it happened. No matrix was rebuilt and no second model was written:
characterization is a separate reading of an inventory whose dates were never
lost.


## 5. The run leaves a record

In [13]:
import pyarrow.parquet as pq

print(report.summary())

out = Path("showcase_log.parquet")
report.log.to_parquet(out)
table = pq.read_table(out)
print()
print(f"wrote {out.name}: {table.num_rows} rows, {table.num_columns} columns")
print("kinds:", sorted(set(table.column("kind").to_pylist())))
out.unlink()

18 nodes, 13 inventory entries
11 unresolved (generalisation_exhausted: 11)
5 proxies (4 incomplete)
attribution: allocation=none, capital=per_output

wrote showcase_log.parquet: 298 rows, 19 columns
kinds: ['attribution', 'biosphere', 'node', 'provenance', 'resolution', 'unresolved']


In [14]:
contributions_figure = viz.contributions(
    assessment, by="node", labels={node.id: node.model for node in report.nodes}
)
contributions_figure

Every node, every cutoff, every parameter fallback and every proxy, one row
each. Parameters arrive as parquet and the whole run leaves as parquet, so two
studies can be diffed with a single read.

## What this changes

- **The supply chain assembles itself.** Models declare vocabulary IRIs, and the
  orchestrator finds who answers what.
- **Missing data is visible.** Cutoffs carry a reason and a position in the
  chain, so a reader can see what a number excludes.
- **Concessions are declared and recorded.** A generalised demand or a borrowed
  dataset is tagged at the node, with what was asked and what answered it.
- **Inventories are time-explicit by construction.** Dates survive the
  traversal, so dynamic characterization needs no second model.
- **The run is a file.** One parquet holds the graph, the gaps and the choices.
- **A process can depend on its demand.** Location, year, scale and feed
  conditions live in the model, where a physical dependency belongs.
- **A model can be a measurement.** Two models, one product, disjoint coverage:
  the year on the demand decides whether you get a meter reading or a
  calculation, and the report says which.

---

- Normative choices, and a model that co-produces: [When a model makes two things](coproduction.ipynb)
- The deeper worked example: [Direct air capture, end to end](dac.ipynb)
- The same pieces in reference form: [Core Concepts](../docs/content/concepts.md)
- [Installation](../docs/content/installation.md)